In [ ]:
package com.example.studentportal;

import jakarta.annotation.PostConstruct;
import jakarta.persistence.*;
import jakarta.validation.Valid;
import jakarta.validation.constraints.*;
import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;
import org.springframework.data.jpa.repository.JpaRepository;
import org.springframework.http.HttpStatus;
import org.springframework.stereotype.Service;
import org.springframework.validation.annotation.Validated;
import org.springframework.web.bind.annotation.*;

import java.util.*;
import java.util.stream.Collectors;

@SpringBootApplication
public class StudentResultPortalApplication {
    public static void main(String[] args) {
        SpringApplication.run(StudentResultPortalApplication.class, args);
    }
}

/*
 Add this to src/main/resources/application.properties:
 spring.datasource.url=jdbc:h2:mem:studentdb;DB_CLOSE_DELAY=-1;MODE=MYSQL
 spring.datasource.driverClassName=org.h2.Driver
 spring.datasource.username=sa
 spring.datasource.password=
 spring.jpa.hibernate.ddl-auto=update
 spring.jpa.show-sql=true
 spring.h2.console.enabled=true
 spring.h2.console.path=/h2-console
 server.port=8080
 */

@Entity
@Table(name = "students")
class Student {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @NotBlank(message = "Name is required")
    @Column(nullable = false)
    private String name;

    @NotBlank(message = "Email is required")
    @Email(message = "Invalid email format")
    @Column(nullable = false, unique = true)
    private String email;

    @NotBlank(message = "Department is required")
    @Column(nullable = false)
    private String department;

    @NotNull(message = "Semester is required")
    @Min(value = 1, message = "Semester must be at least 1")
    @Column(nullable = false)
    private Integer semester;

    @OneToMany(mappedBy = "student", cascade = CascadeType.ALL, orphanRemoval = true)
    private List<ResultRecord> results = new ArrayList<>();

    public Student() {
    }

    public Student(String name, String email, String department, Integer semester) {
        this.name = name;
        this.email = email;
        this.department = department;
        this.semester = semester;
    }

    public Long getId() {
        return id;
    }

    public void setId(Long id) {
        this.id = id;
    }

    public String getName() {
        return name;
    }

    public void setName(String name) {
        this.name = name;
    }

    public String getEmail() {
        return email;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public String getDepartment() {
        return department;
    }

    public void setDepartment(String department) {
        this.department = department;
    }

    public Integer getSemester() {
        return semester;
    }

    public void setSemester(Integer semester) {
        this.semester = semester;
    }

    public List<ResultRecord> getResults() {
        return results;
    }

    public void setResults(List<ResultRecord> results) {
        this.results = results;
    }
}

@Entity
@Table(name = "results")
class ResultRecord {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @NotBlank(message = "Subject is required")
    @Column(nullable = false)
    private String subject;

    @NotNull(message = "Marks are required")
    @Min(value = 0, message = "Marks cannot be negative")
    @Max(value = 100, message = "Marks cannot exceed 100")
    @Column(nullable = false)
    private Integer marks;

    @ManyToOne(fetch = FetchType.LAZY)
    @JoinColumn(name = "student_id", nullable = false)
    private Student student;

    public ResultRecord() {
    }

    public ResultRecord(String subject, Integer marks, Student student) {
        this.subject = subject;
        this.marks = marks;
        this.student = student;
    }

    public Long getId() {
        return id;
    }

    public void setId(Long id) {
        this.id = id;
    }

    public String getSubject() {
        return subject;
    }

    public void setSubject(String subject) {
        this.subject = subject;
    }

    public Integer getMarks() {
        return marks;
    }

    public void setMarks(Integer marks) {
        this.marks = marks;
    }

    public Student getStudent() {
        return student;
    }

    public void setStudent(Student student) {
        this.student = student;
    }

    public String getGrade() {
        if (marks == null) return "N/A";
        if (marks >= 90) return "A+";
        if (marks >= 80) return "A";
        if (marks >= 70) return "B";
        if (marks >= 60) return "C";
        if (marks >= 50) return "D";
        return "F";
    }

    public boolean isPass() {
        return marks != null && marks >= 40;
    }
}

interface StudentRepository extends JpaRepository<Student, Long> {
    Optional<Student> findByEmail(String email);
}

interface ResultRepository extends JpaRepository<ResultRecord, Long> {
    List<ResultRecord> findByStudentId(Long studentId);
}

@Service
class StudentResultService {
    private final StudentRepository studentRepository;
    private final ResultRepository resultRepository;

    public StudentResultService(StudentRepository studentRepository, ResultRepository resultRepository) {
        this.studentRepository = studentRepository;
        this.resultRepository = resultRepository;
    }

    public Student createStudent(Student student) {
        return studentRepository.save(student);
    }

    public List<Student> getAllStudents() {
        return studentRepository.findAll();
    }

    public Optional<Student> getStudentById(Long id) {
        return studentRepository.findById(id);
    }

    public ResultRecord addResult(Long studentId, ResultRecord result) {
        Student student = studentRepository.findById(studentId)
                .orElseThrow(() -> new NoSuchElementException("Student not found with id: " + studentId));
        result.setStudent(student);
        return resultRepository.save(result);
    }

    public List<ResultRecord> getResultsByStudent(Long studentId) {
        return resultRepository.findByStudentId(studentId);
    }

    public StudentResultSummary getStudentSummary(Long studentId) {
        Student student = studentRepository.findById(studentId)
                .orElseThrow(() -> new NoSuchElementException("Student not found with id: " + studentId));

        List<ResultRecord> results = resultRepository.findByStudentId(studentId);

        int totalMarks = results.stream().mapToInt(ResultRecord::getMarks).sum();
        double percentage = results.isEmpty() ? 0 : (double) totalMarks / results.size();
        String overallGrade = calculateGrade(percentage);
        long totalSubjects = results.size();
        long passedSubjects = results.stream().filter(ResultRecord::isPass).count();

        return new StudentResultSummary(
                student.getId(),
                student.getName(),
                student.getDepartment(),
                student.getSemester(),
                totalSubjects,
                passedSubjects,
                totalMarks,
                Math.round(percentage * 100.0) / 100.0,
                overallGrade
        );
    }

    private String calculateGrade(double percentage) {
        if (percentage >= 90) return "A+";
        if (percentage >= 80) return "A";
        if (percentage >= 70) return "B";
        if (percentage >= 60) return "C";
        if (percentage >= 50) return "D";
        return "F";
    }

    public void seedData() {
        if (studentRepository.count() == 0) {
            Student s1 = new Student("Alice Johnson", "alice@example.com", "Computer Science", 6);
            Student s2 = new Student("Bob Smith", "bob@example.com", "Electronics", 4);

            s1 = studentRepository.save(s1);
            s2 = studentRepository.save(s2);

            resultRepository.save(new ResultRecord("Java", 92, s1));
            resultRepository.save(new ResultRecord("Database", 88, s1));
            resultRepository.save(new ResultRecord("Math", 75, s1));

            resultRepository.save(new ResultRecord("Circuit Theory", 82, s2));
            resultRepository.save(new ResultRecord("Signals", 71, s2));
            resultRepository.save(new ResultRecord("Physics", 49, s2));
        }
    }
}

@RestController
@RequestMapping("/api")
@Validated
class StudentResultController {
    private final StudentResultService service;

    public StudentResultController(StudentResultService service) {
        this.service = service;
    }

    @GetMapping("/students")
    public List<StudentResponse> getAllStudents() {
        return service.getAllStudents().stream()
                .map(StudentResponse::from)
                .collect(Collectors.toList());
    }

    @GetMapping("/students/{id}")
    public StudentResponse getStudentById(@PathVariable Long id) {
        return service.getStudentById(id)
                .map(StudentResponse::from)
                .orElseThrow(() -> new StudentNotFoundException(id));
    }

    @PostMapping("/students")
    @ResponseStatus(HttpStatus.CREATED)
    public StudentResponse createStudent(@Valid @RequestBody StudentRequest request) {
        Student student = new Student(request.name(), request.email(), request.department(), request.semester());
        return StudentResponse.from(service.createStudent(student));
    }

    @PostMapping("/students/{id}/results")
    @ResponseStatus(HttpStatus.CREATED)
    public ResultResponse addResult(@PathVariable Long id, @Valid @RequestBody ResultRequest request) {
        ResultRecord record = new ResultRecord(request.subject(), request.marks(), null);
        ResultRecord saved = service.addResult(id, record);
        return ResultResponse.from(saved);
    }

    @GetMapping("/students/{id}/results")
    public List<ResultResponse> getResults(@PathVariable Long id) {
        return service.getResultsByStudent(id).stream()
                .map(ResultResponse::from)
                .collect(Collectors.toList());
    }

    @GetMapping("/students/{id}/summary")
    public StudentResultSummary getSummary(@PathVariable Long id) {
        return service.getStudentSummary(id);
    }

    @ExceptionHandler(NoSuchElementException.class)
    @ResponseStatus(HttpStatus.NOT_FOUND)
    public ErrorResponse handleNotFound(NoSuchElementException ex) {
        return new ErrorResponse("NOT_FOUND", ex.getMessage());
    }

    @ExceptionHandler(Exception.class)
    @ResponseStatus(HttpStatus.BAD_REQUEST)
    public ErrorResponse handleBadRequest(Exception ex) {
        return new ErrorResponse("BAD_REQUEST", ex.getMessage());
    }
}

record StudentRequest(
        @NotBlank String name,
        @NotBlank @Email String email,
        @NotBlank String department,
        @NotNull @Min(1) Integer semester
) {}

record StudentResponse(Long id, String name, String email, String department, Integer semester) {
    static StudentResponse from(Student student) {
        return new StudentResponse(student.getId(), student.getName(), student.getEmail(), student.getDepartment(), student.getSemester());
    }
}

record ResultRequest(
        @NotBlank String subject,
        @NotNull @Min(0) @Max(100) Integer marks
) {}

record ResultResponse(Long id, String subject, Integer marks, String grade, boolean pass) {
    static ResultResponse from(ResultRecord result) {
        return new ResultResponse(result.getId(), result.getSubject(), result.getMarks(), result.getGrade(), result.isPass());
    }
}

record StudentResultSummary(
        Long studentId,
        String studentName,
        String department,
        Integer semester,
        Long totalSubjects,
        Long passedSubjects,
        Integer totalMarks,
        Double percentage,
        String overallGrade
) {}

record ErrorResponse(String code, String message) {}

class StudentNotFoundException extends RuntimeException {
    public StudentNotFoundException(Long id) {
        super("Student not found with id: " + id);
    }
}

@Component
class DataInitializer {
    private final StudentResultService studentResultService;

    public DataInitializer(StudentResultService studentResultService) {
        this.studentResultService = studentResultService;
    }

    @PostConstruct
    public void init() {
        studentResultService.seedData();
    }
}